# Validación cruzada — `nairu_dataset.csv` vs `nairu_estimates_v6.csv`

**Objetivo:** verificar que las series de insumo de mi pipeline (`nairu_dataset.csv`) coinciden con las que un compañero usó en su modelo de NAIRU (`nairu_estimates_v6.csv`).

Si ambas series de desempleo y de capacidad utilizada coinciden razonablemente en su período común, podemos asumir que mi base reproduce sus insumos y que las diferencias entre estimaciones de NAIRU se deben al modelo, no al input.

**Variables comunes entre ambas bases:**

| Mi pipeline (`nairu_dataset.csv`) | Compañero (`nairu_estimates_v6.csv`) | Concepto |
|---|---|---|
| `unemployment_rate` | `unemployment_current` | Tasa de desempleo nacional (DANE GEIH desest.) |
| `capacity_utilization` | `icu_current` | Capacidad utilizada industrial (ANDI EOIC) |
| `Inf_Rate` | (no directo, usa `inflation_gap`) | Inflación observada |

El compañero también incluye variables derivadas (lags, hysteresis, oil shock) y el output del modelo (`nairu_estimate`, `naicu_estimate`, intervalos de confianza).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MINE  = ROOT / "data" / "final"  / "nairu_dataset.csv"
PEER  = ROOT / "data" / "inputs" / "nairu_estimates_v6.csv"

mine = pd.read_csv(MINE,  parse_dates=["date"]).set_index("date").sort_index()
peer = pd.read_csv(PEER,  parse_dates=["Date"]).set_index("Date").sort_index()
peer.index.name = "date"

print(f"Mi base : {len(mine):>4} filas, {mine.index.min().date()} → {mine.index.max().date()}")
print(f"Peer    : {len(peer):>4} filas, {peer.index.min().date()} → {peer.index.max().date()}")
common = mine.index.intersection(peer.index)
print(f"Común   : {len(common):>4} fechas, {common.min().date()} → {common.max().date()}")

## 1. Desempleo — `unemployment_rate` vs `unemployment_current`

Si ambas series provienen del mismo anexo del DANE (GEIH desestacionalizado, Total Nacional), deberían ser idénticas en el período común. Toleramos diferencias de redondeo (~0.05pp) por revisiones del DANE.

In [ ]:
u_mine = mine.loc[common, "unemployment_rate"].rename("mine")
u_peer = peer.loc[common, "unemployment_current"].rename("peer")

u = pd.concat([u_mine, u_peer], axis=1).dropna()
u["diff"] = u["mine"] - u["peer"]

print("Estadísticas de la diferencia (mine - peer):")
print(u["diff"].describe().round(3))
print()
if u["diff"].abs().max() < 0.5:
    print("OK — diferencias menores a 0.5pp en todo el período común.")
else:
    n_bad = (u["diff"].abs() >= 0.5).sum()
    print(f"Atención: {n_bad} observaciones con |diff| ≥ 0.5pp.")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                         gridspec_kw={"height_ratios": [3, 1]})
u[["mine", "peer"]].plot(ax=axes[0], lw=1)
axes[0].set_ylabel("Tasa de desempleo (%)")
axes[0].set_title("Comparación de desempleo: mi pipeline vs base del compañero")
axes[0].grid(alpha=0.3)
axes[0].legend(["mine (DANE GEIH)", "peer (v6)"])

u["diff"].plot(ax=axes[1], color="crimson", lw=0.7)
axes[1].axhline(0, color="black", lw=0.5)
axes[1].set_ylabel("diff (pp)")
axes[1].grid(alpha=0.3)
plt.tight_layout()

## 2. Capacidad utilizada — `capacity_utilization` vs `icu_current`

Mi serie viene del scraping de los PDFs de la **EOIC de ANDI**; la del compañero se nombra ``icu_current`` (Industrial Capacity Utilization). En principio deberían coincidir, pero ANDI cambia metodología cada cierto tiempo y mi pipeline empieza solo en 2017.

In [ ]:
icu_mine = mine.loc[common, "capacity_utilization"].rename("mine")
icu_peer = peer.loc[common, "icu_current"].rename("peer")
icu = pd.concat([icu_mine, icu_peer], axis=1).dropna()
icu["diff"] = icu["mine"] - icu["peer"]

print("Período común con datos en ambas:", len(icu), "obs")
print("Estadísticas de diff (mine - peer):")
print(icu["diff"].describe().round(3))

In [ ]:
if len(icu) > 0:
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                             gridspec_kw={"height_ratios": [3, 1]})
    icu[["mine", "peer"]].plot(ax=axes[0], lw=1)
    axes[0].set_ylabel("Capacidad utilizada (%)")
    axes[0].set_title("Capacidad utilizada industrial (ANDI EOIC): mi pipeline vs compañero")
    axes[0].grid(alpha=0.3)
    axes[0].legend(["mine (ANDI EOIC PDF)", "peer (v6)"])

    icu["diff"].plot(ax=axes[1], color="crimson", lw=0.7)
    axes[1].axhline(0, color="black", lw=0.5)
    axes[1].set_ylabel("diff (pp)")
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
else:
    print("Sin solapamiento temporal entre las dos series.")

## 3. Estimaciones de NAIRU del compañero

Visualizamos `nairu_estimate` con sus intervalos de confianza al 90% y 95% (output del modelo del compañero), junto con el desempleo observado. El **gap de desempleo** (= u − NAIRU) es la variable cíclica clave.

In [ ]:
needed = ["unemployment_current", "nairu_estimate",
          "nairu_ci_lower_90", "nairu_ci_upper_90",
          "nairu_ci_lower_95", "nairu_ci_upper_95"]
if all(c in peer.columns for c in needed):
    p = peer[needed].dropna()

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.fill_between(p.index, p["nairu_ci_lower_95"], p["nairu_ci_upper_95"],
                    color="steelblue", alpha=0.15, label="NAIRU IC 95%")
    ax.fill_between(p.index, p["nairu_ci_lower_90"], p["nairu_ci_upper_90"],
                    color="steelblue", alpha=0.30, label="NAIRU IC 90%")
    ax.plot(p.index, p["nairu_estimate"], color="steelblue", lw=1.5, label="NAIRU estimada")
    ax.plot(p.index, p["unemployment_current"], color="crimson", lw=1, label="Desempleo observado")
    ax.set_ylabel("%")
    ax.set_title("NAIRU del compañero (v6) vs desempleo observado")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
else:
    print("Faltan columnas en peer:", set(needed) - set(peer.columns))

## 4. Resumen de diferencias

Tabla compacta para auditoría: si las diferencias son grandes, hay que revisar de dónde viene cada serie.

In [ ]:
summary = []
pairs = [
    ("Desempleo",            "unemployment_rate",   "unemployment_current"),
    ("Capacidad utilizada",  "capacity_utilization", "icu_current"),
]
for label, mine_col, peer_col in pairs:
    if mine_col not in mine.columns or peer_col not in peer.columns:
        continue
    paired = pd.concat([
        mine.loc[common, mine_col].rename("mine"),
        peer.loc[common, peer_col].rename("peer"),
    ], axis=1).dropna()
    diff = paired["mine"] - paired["peer"]
    summary.append({
        "variable":     label,
        "obs comunes":  len(paired),
        "mean_diff":    round(diff.mean(), 3),
        "median_diff":  round(diff.median(), 3),
        "std_diff":     round(diff.std(), 3),
        "max_abs_diff": round(diff.abs().max(), 3),
        "corr":         round(paired.corr().iloc[0, 1], 4),
    })
pd.DataFrame(summary).set_index("variable")

---

**Lectura:**

- `corr` cercano a 1 y `max_abs_diff` pequeño → mi pipeline reproduce los insumos del compañero.
- `corr` < 0.95 o `max_abs_diff` > 0.5pp → revisar la fuente y las transformaciones.
- En `capacity_utilization`, esperamos diferencias mayores porque ANDI publica revisiones y mi scraper procesa los PDFs originales — vale la pena auditar los puntos más divergentes.